# Attenomy ML Internship : Week 1 : Task 1.2
## Data Cleaning, Missingness Imputation & Schema Normalization

**Objective:** Ingest the raw benchmark telemetry dataset (`data/raw/attenomy_telemetry_raw.csv`), audit data hygiene, resolve missing values, eliminate anomalies, and export clean data to `data/clean/clean.parquet`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (  # noqa: E402
    load_data,
    summarize_dataframe_health,
    train_val_test_split,
)

print("Project root:", PROJECT_ROOT)
print("Environment and dependencies initialized successfully.")

Project root: e:\ml-engineer-internship
Environment and dependencies initialized successfully.


### 1. Ingest Raw Telemetry Dataset

In [2]:
raw = pd.read_csv("../data/raw/attenomy_telemetry_raw.csv")
print(raw.shape)
print(raw.isnull().sum())
print("negative durations:", (raw["session_duration_sec"] < 0).sum())
print("duplicate rows:", raw.duplicated().sum())
print("duplicate transaction_id:", raw["transaction_id"].duplicated().sum())
bad_ids = ~raw["transaction_id"].astype(str).str.fullmatch(r"TRX_\d{6}")
print("malformed transaction_id:", bad_ids.sum())
ts = pd.to_datetime(raw["timestamp"], errors="coerce", utc=True)
print("unparseable timestamps:", ts.isna().sum())
for col in ["interaction_type", "channel", "client_tier", "region", "error_code"]:
    print(raw[col].value_counts(dropna=False), "\n")
print(raw.describe().T)
print(raw["is_anomaly"].value_counts(normalize=True))

(25000, 13)
transaction_id              0
account_id                  0
timestamp                   0
interaction_type            0
channel                     0
session_duration_sec      972
payload_size_kb             0
latency_ms                  0
retry_count                 0
error_code              22954
client_tier               809
region                      0
is_anomaly                  0
dtype: int64
negative durations: 60
duplicate rows: 0
duplicate transaction_id: 0
malformed transaction_id: 0
unparseable timestamps: 0
interaction_type
api_query              8817
doc_extraction         6208
agent_orchestration    4503
model_inference        3686
batch_sync             1786
Name: count, dtype: int64 

channel
web_app       8663
python_sdk    6371
rest_api      4972
cli_tool      2522
WEB_APP        991
REST_API       748
Python_SDK     733
Name: count, dtype: int64 

client_tier
Starter       10877
Pro            8388
Enterprise     3692
VIP            1234
NaN             

In [3]:
REQUIRED_COLUMNS = [
    "transaction_id", "account_id", "timestamp", "interaction_type", "channel",
    "session_duration_sec", "payload_size_kb", "latency_ms", "retry_count",
    "error_code", "client_tier", "region", "is_anomaly",
]
raw_path = '../data/raw/attenomy_telemetry_raw.csv'
df = load_data(raw_path, required_columns=REQUIRED_COLUMNS)
health = summarize_dataframe_health(df)
print(f'Rows: {health["row_count"]:,}')
print(f'Columns: {health["column_count"]}')
print(f'Memory Footprint: {health["memory_mb"]} MB')
print(f'Duplicate Rows: {health["duplicate_rows"]}')

Rows: 25,000
Columns: 13
Memory Footprint: 12.27 MB
Duplicate Rows: 0


In [4]:
valid_id = df["transaction_id"].str.fullmatch(r"TRX_\d{6}")
print("corrupted primary keys dropped:", int((~valid_id).sum()))
df = df[valid_id].copy()
print("duplicate transaction_ids dropped:", int(df["transaction_id"].duplicated().sum()))
df = df.drop_duplicates(subset="transaction_id", keep="first")


for col in ["interaction_type", "channel", "region"]:
    df[col] = df[col].str.lower()


df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
print("unparseable timestamps dropped:", int(df["timestamp"].isna().sum()))
df = df[df["timestamp"].notna()].copy()


neg = df["session_duration_sec"] < 0
print("negative durations set to missing:", int(neg.sum()))
df.loc[neg, "session_duration_sec"] = np.nan


df["error_code"] = df["error_code"].replace("UNKNOWN", "NO_ERROR")

print(df.shape)
print(df["channel"].value_counts(), "\n")
print(df["client_tier"].value_counts(), "\n")
print(df["error_code"].value_counts())
print("remaining nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])

corrupted primary keys dropped: 0
duplicate transaction_ids dropped: 0
unparseable timestamps dropped: 0
negative durations set to missing: 60
(25000, 13)
channel
web_app       9654
python_sdk    7104
rest_api      5720
cli_tool      2522
Name: count, dtype: int64 

client_tier
Starter       10877
Pro            8388
Enterprise     3692
VIP            1234
UNKNOWN         809
Name: count, dtype: int64 

error_code
NO_ERROR                    22954
ERR_AUTH_EXPIRED              431
ERR_RATE_LIMIT                420
ERR_PAYLOAD_TOO_LARGE         411
ERR_UPSTREAM_UNAVAILABLE      398
ERR_TIMEOUT                   386
Name: count, dtype: int64
remaining nulls:
 session_duration_sec    1032
dtype: int64


### 2. Missing Value Audit & Imputation Strategy

| Column | Issue | Strategy | Fit on |
|---|---|---|---|
| session_duration_sec | 972 nulls + 60 negative values converted to nulls | Median imputation | Training set only |
| client_tier | 809 missing values | Fill with `UNKNOWN` | No fitting required |
| error_code | 22,954 missing values | Fill with `NO_ERROR`, if supported by the dataset definition | No fitting required |
| transaction_id | Invalid or missing identifiers | Drop invalid rows | No fitting required |

**Note:** The audit uses the original raw dataset so that missing categorical values remain visible before normalization.

In [5]:
null_summary = raw.isnull().sum()

audit = pd.DataFrame({
    "null_count": null_summary,
    "null_pct": (null_summary / len(raw) * 100).round(2),
})

display(audit[audit["null_count"] > 0])

,null_count,null_pct
session_duration_sec,972,3.89
error_code,22954,91.82
client_tier,809,3.24


In [6]:
train_df, val_df, test_df = train_val_test_split(
    df,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    stratify_col="is_anomaly",
    seed=42,
)

print("train / val / test:", len(train_df), len(val_df), len(test_df))
print("sum equals total:", len(train_df) + len(val_df) + len(test_df) == len(df))
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name} anomaly rate: {part['is_anomaly'].mean():.4%}")
print("disjoint:",
      set(train_df.index).isdisjoint(val_df.index)
      and set(train_df.index).isdisjoint(test_df.index)
      and set(val_df.index).isdisjoint(test_df.index))

train / val / test: 17500 3750 3750
sum equals total: True
train anomaly rate: 3.9143%
val anomaly rate: 3.9200%
test anomaly rate: 3.9200%
disjoint: True


### 4. Imputation & Outlier Handling (after the split, fit on train only)

- Median of session_duration_sec = 130.6, learned from train rows and applied to val and test.
- IQR cap on session_duration_sec: upper bound 520.05 (train only). Values above it are capped, not deleted.
- latency_ms and payload_size_kb are NOT capped: their spikes may be the real anomaly signal. Risk noted for the memo; Week 2 will compare capped vs uncapped.

In [7]:
median_duration = train_df["session_duration_sec"].median()
print("train median session_duration_sec:", round(median_duration, 2))

for part in (train_df, val_df, test_df):
    part["session_duration_sec"] = part["session_duration_sec"].fillna(median_duration)

# IQR bounds come from TRAIN rows only
splits = [("train", train_df), ("val", val_df), ("test", test_df)]
bounds = {}
for col in ["session_duration_sec", "payload_size_kb", "latency_ms"]:
    q1, q3 = train_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    bounds[col] = (max(q1 - 1.5 * iqr, 0), q3 + 1.5 * iqr)
    lo, hi = bounds[col]
    counts = {name: int(((part[col] < lo) | (part[col] > hi)).sum()) for name, part in splits}
    print(f"{col}: bounds=({lo:.2f}, {hi:.2f}), outliers={counts}")

# Cap only session_duration_sec; keep latency and payload spikes as real signal
lo, hi = bounds["session_duration_sec"]
for part in (train_df, val_df, test_df):
    part["session_duration_sec"] = part["session_duration_sec"].clip(lower=lo, upper=hi)

for name, part in splits:
    print(name, "nulls:", int(part.isnull().sum().sum()))

train median session_duration_sec: 130.6
session_duration_sec: bounds=(0.00, 520.05), outliers={'train': 784, 'val': 144, 'test': 180}
payload_size_kb: bounds=(0.00, 75.69), outliers={'train': 1446, 'val': 289, 'test': 310}
latency_ms: bounds=(0.00, 166.95), outliers={'train': 311, 'val': 65, 'test': 70}
train nulls: 0
val nulls: 0
test nulls: 0


### 5. Encoding & Scaling Specification (to be fit on TRAIN only in Week 2)

| Column(s) | Type | Planned transform |
|---|---|---|
| interaction_type, channel, region | Nominal | One-hot encoding |
| client_tier | Ordinal | Starter < Pro < Enterprise < VIP, with UNKNOWN as its own category |
| error_code | Nominal | One-hot encoding (NO_ERROR is the baseline) |
| session_duration_sec, payload_size_kb, latency_ms | Continuous | StandardScaler |
| retry_count | Count | StandardScaler |
| transaction_id, account_id | Identifiers | Dropped before modeling |
| timestamp | Datetime | Extract hour and weekday for EDA |

### 6. Export Cleaned Parquet Dataset

Combine the train, validation, and test splits into one cleaned dataset, preserve the split labels, validate the final data, and export it as a Snappy-compressed Parquet file.

In [10]:
clean = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    test_df.assign(split="test"),
]).sort_index()

for col in ["interaction_type", "channel", "client_tier", "region", "error_code", "split"]:
    clean[col] = clean[col].astype("category")
clean["retry_count"] = clean["retry_count"].astype("int32")
clean["is_anomaly"] = clean["is_anomaly"].astype("int8")

assert int(clean.isnull().sum().sum()) == 0, "nulls remain"
assert (clean["session_duration_sec"] >= 0).all(), "negative durations remain"
assert clean["transaction_id"].is_unique, "duplicate keys"

clean_path = PROJECT_ROOT / "data" / "clean" / "clean.parquet"
clean_path.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(clean_path, compression="snappy", index=False)

check = pd.read_parquet(clean_path)
print("saved:", clean_path)
print("size (MB):", round(clean_path.stat().st_size / 1024 / 1024, 2))
print("shape:", check.shape, "| nulls:", int(check.isnull().sum().sum()))
print(check["split"].value_counts())
print(check.dtypes)

saved: e:\ml-engineer-internship\data\clean\clean.parquet
size (MB): 0.65
shape: (25000, 14) | nulls: 0
split
train    17500
test      3750
val       3750
Name: count, dtype: int64
transaction_id                          str
account_id                              str
timestamp               datetime64[us, UTC]
interaction_type                   category
channel                            category
session_duration_sec                float64
payload_size_kb                     float64
latency_ms                          float64
retry_count                           int32
error_code                         category
client_tier                        category
region                             category
is_anomaly                             int8
split                              category
dtype: object


In [9]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

PROJECT_ROOT: e:\ml-engineer-internship
Train shape: (17500, 13)
Validation shape: (3750, 13)
Test shape: (3750, 13)
